> **Catatan (diperbarui):** notebook ini adalah catatan eksplorasi dan analisis awal. Pipeline final yang dipakai aplikasi ada di folder `src/` dan dijalankan lewat `retrain.py`. Angka final (MAPE, rentang 80%, dan pilihan model vs harga bulan lalu per komoditas) ada di folder `results/`. Beberapa sel di bawah mungkin menggunakan kriteria seleksi lama (MAE) dan fitur lama (`geo_id`, `month`).

# 01 - Audit Dataset & EDA
Dataset World Bank RTFP Indonesia. Semua angka dihitung dari file aktual.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd, numpy as np
from IPython.display import Image, display
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

In [ ]:
from src.data_loader import load_raw
raw = load_raw()
print(raw.shape)
raw.dtypes.value_counts()

In [ ]:
info = pd.DataFrame({'dtype': raw.dtypes.astype(str), 'missing': raw.isna().sum(), 'missing_%': (raw.isna().mean()*100).round(2), 'unik': raw.nunique()})
info

In [ ]:
print('Duplikasi baris:', raw.duplicated().sum(), '| duplikasi (pasar, bulan):', raw.duplicated(['geo_id','price_date']).sum())
print('Pasar:', raw.geo_id.nunique(), '| Provinsi:', raw.adm1_name.nunique(), '| Kab/Kota:', raw.adm2_name.nunique())
print('Periode:', raw.price_date.min(), '-', raw.price_date.max(), '| spatially_interpolated:', raw.spatially_interpolated.unique(), '| last_survey_point:', raw.last_survey_point.unique())

**Temuan kunci:** kolom tanpa prefiks (mis. `rice`) = harga observasi; `c_*` = *close estimate* (estimasi model World Bank, terisi penuh). Hanya ~32% sel merupakan observasi, dan observasi padat baru ada sejak 2020. Baris `gid_idn_national_average` adalah agregat nasional (bukan pasar) dan dikeluarkan. Oktober-November 2026 tidak memiliki observasi (last survey point = Sep 2026).

In [ ]:
from src.preprocessing import build_long
long = build_long(raw)
long.groupby('year')['is_observed'].mean().round(3)

In [ ]:
from src.eda import make_eda_figures
s = make_eda_figures(raw, long)
pd.DataFrame(s['per_commodity']).T.round(1)

In [ ]:
for f in sorted(Path('results/figures').glob('eda_*.png')):
    display(Image(str(f)))

## Kesimpulan EDA
1. 53.297 baris x 74 kolom, 223 `geo_id` = 222 pasar + 1 baris agregat 'Market Average' (rata-rata nasional, dikeluarkan dari analisis pasar), 35 provinsi, 239 bulan (Jan 2007-Nov 2026), tanpa duplikasi.
2. Panel seimbang, tetapi hanya ~32% sel berisi harga observasi; sisanya estimasi model.
3. Observasi pasar ada sejak 2016 (2017 nyaris kosong, 2018-2019 tidak ada); padat Jan 2020-Sep 2026 (celah: Feb-Mei 2024 dan Jan 2026).
4. 30 observasi ekstrem (rasio terhadap estimasi di luar 0,5x-2x) ditandai outlier dan dikecualikan dari pemodelan (tidak dihapus dari dataset).
5. `spatially_interpolated` bernilai 0 untuk seluruh baris.